# B3 · Cost-complexity pruning chosen in cross-validation

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Use `cost_complexity_pruning_path` and `ccp_alpha` to prune a regression tree on the house prices, choose `alpha` in cross-validation (with the pruning **inside** each fold) applying both the minimum rule and the "one standard deviation" rule (the smallest tree within
one standard error of the best), and compare the number of leaves and the error.

## Dataset

`ds.california()`, 75/25 split with `random_state=0`; base tree `DecisionTreeRegressor(min_samples_leaf=5, random_state=0)`; validation `KFold(5, shuffle=True, random_state=0)`.

## Educational purpose

Understand how `ccp_alpha` turns the complexity/error trade-off into a single continuous parameter, and learn the parsimony rule (1-SE) to choose the simplest model that cannot be told apart from the best.

## Task

1. Train the base tree on the training set and save `n_alphas` (number of values in `cost_complexity_pruning_path(...).ccp_alphas`) and `initial_leaves` (`get_n_leaves()` of the unpruned tree).
2. For `alphas = np.geomspace(1e-5, 1e-2, 30)` compute in cross-validation (pruning inside each fold: pass `ccp_alpha` to the estimator) `cv_rmse` (mean RMSE, `pd.Series` indexed by `alphas`) and `cv_se` (standard error = standard deviation of the fold RMSEs / √5), and `leaves` (number of leaves of the tree pruned with that `alpha` trained on the whole training set).
3. Save `alpha_min` (the `alpha` with minimum RMSE) and `alpha_1se`: the **largest** `alpha` with `cv_rmse <= cv_rmse.min() + cv_se[alpha_min]`.
4. On the test set save `rmse_min` and `rmse_1se` (trees trained with those `alpha` on the full training set) and `leaves_min`, `leaves_1se`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)
base = DecisionTreeRegressor(min_samples_leaf=5, random_state=0)
alphas = np.geomspace(1e-5, 1e-2, 30)

## Your work

In [ ]:
n_alphas = None
initial_leaves = None

In [ ]:
cv_rmse = pd.Series(dtype=float)
cv_se = pd.Series(dtype=float)
leaves = pd.Series(dtype=float)

In [ ]:
alpha_min = None
alpha_1se = None

In [ ]:
rmse_min = None
rmse_1se = None
leaves_min = None
leaves_1se = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`scores = -cross_val_score(DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=a), X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error")`; `cv_rmse[a] = scores.mean()`; `cv_se[a] = scores.std(ddof=1) / np.sqrt(5)`. `alpha_1se = max(a for a in alphas if cv_rmse[a] <= cv_rmse.min() + cv_se[alpha_min])`.'):
    check(n_alphas is not None and n_alphas > 1500 and initial_leaves > 2000, "the base tree has over 2000 leaves and over 1500 candidate alpha values", "cost_complexity_pruning_path and get_n_leaves")
    check(len(cv_rmse) == 30 and len(cv_se) == 30 and len(leaves) == 30 and leaves.is_monotonic_decreasing and leaves.iloc[0] > 1000 and leaves.iloc[-1] < 30, "30 alpha values; the leaves decrease (from over 1000 to under 30)", "more alpha, more pruning")
    check(1e-4 <= alpha_min <= 1e-3 and alpha_1se >= alpha_min and cv_rmse[alpha_1se] <= cv_rmse.min() + cv_se[alpha_min] + 1e-12, "best alpha between 1e-4 and 1e-3; 1-SE alpha >= alpha_min and within one standard error", "the largest alpha within cv_rmse.min() + cv_se[alpha_min]")
    check(leaves_1se <= leaves_min and leaves_min < initial_leaves / 5, "the 1-SE tree has at most the leaves of the minimum one; both well below the initial leaves", "get_n_leaves()")
    check(rmse_1se < rmse_min + 0.03 and rmse_min < 0.66, "the 1-SE tree loses less than 0.03 of RMSE compared with the minimum (RMSE below 0.66)", "compare rmse_min and rmse_1se")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_cost_complexity_pruning_in_cv.ipynb`. Open it after you pass the check, or after you have really tried.